In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use("ggplot")

TICKERS = ["SPY", "TLT", "GLD", "^VIX"]

prices = yf.download(
    TICKERS,
    start="2005-01-01",
    end="2026-06-30",
    auto_adjust=True,
    progress=False)["Close"]

prices = prices.dropna()
log_returns = np.log(prices / prices.shift(1))
log_returns = log_returns.dropna()

rolling_vol = log_returns["SPY"].rolling(window=63).std() * np.sqrt(252)

rolling_mean = log_returns["SPY"].rolling(window=252).mean()

rolling_var = log_returns["SPY"].rolling(window=252).var()

In [ ]:
features = log_returns.copy()

In [ ]:
features.columns = [
    "GLD_Return",
    "SPY_Return",
    "TLT_Return",
    "VIX_Return"]

In [ ]:
# 20-day rolling volatility
rolling_vol_20 = features.rolling(window=20).std()

In [ ]:
rolling_vol_20.columns = ["GLD_Vol20","SPY_Vol20","TLT_Vol20","VIX_Vol20"]

In [ ]:
features = pd.concat([features, rolling_vol_20],axis=1)

In [ ]:
features = features.dropna()

In [ ]:
rolling_vol_63 = (features[["GLD_Return", "SPY_Return", "TLT_Return", "VIX_Return"]].rolling(window=63).std())
rolling_vol_63.columns = ["GLD_Vol63","SPY_Vol63","TLT_Vol63","VIX_Vol63",]

In [ ]:
features = pd.concat([features, rolling_vol_63],axis=1)

In [ ]:
features = features.dropna()

In [ ]:
# 20-Day Momentum
momentum_20 = (prices.pct_change(periods=20))

In [ ]:
momentum_63 = (prices.pct_change(periods=63))
momentum_63.columns = [
    "GLD_Mom63",
    "SPY_Mom63",
    "TLT_Mom63",
    "VIX_Mom63"]

In [ ]:
momentum_20 = momentum_20.loc[features.index]
momentum_63 = momentum_63.loc[features.index]

In [ ]:
features = pd.concat([features,momentum_20,momentum_63],axis=1)
features = features.dropna()

In [ ]:
rolling_corr_spy_tlt = (
    features["SPY_Return"]
    .rolling(window=20)
    .corr(features["TLT_Return"]))

rolling_corr_spy_tlt.name = "SPY_TLT_Corr20"

In [ ]:
features = pd.concat([features, rolling_corr_spy_tlt],axis=1)
features = features.dropna()

In [ ]:
vix_level = prices["^VIX"].loc[features.index]
features["VIX_Level"] = vix_level

In [ ]:
# Load SPY returns from your feature matrix
spy_returns = features["SPY"]

# Count consecutive positive and negative return streaks
sign = (spy_returns > 0).astype(int)

streaks = []
current = 1

for i in range(1, len(sign)):
    if sign.iloc[i] == sign.iloc[i - 1]:
        current += 1
    else:
        streaks.append(current)
        current = 1

streaks.append(current)

print(f"Average streak length: {sum(streaks) / len(streaks):.2f} days")
print(f"Maximum streak length: {max(streaks)} days")

Average streak length: 11.10 days
Maximum streak length: 108 days
